# Package

In [53]:
# ==========================================
# Chargement
# ==========================================
import os
import pickle
import numpy as np
import pandas as pd

from __future__ import annotations
from typing import Any, Callable, Dict, Iterable, List, Optional, Tuple, Union

from dateutil.relativedelta import relativedelta

import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Chargement des résultats

In [54]:
# ---------- Helpers ----------

def load_any(path):
    """Charge un objet avec joblib puis pickle si besoin."""
    if not os.path.exists(path):
        raise FileNotFoundError(f"Fichier introuvable: {path}")
    try:
        return joblib.load(path)
    except Exception:
        with open(path, "rb") as f:
            return pickle.load(f)

def load_with_fallbacks(primary_path, *alt_paths, expect_type=None, label=None):
    """
    Charge primary_path puis essaie les alt_paths en fallback.
    - expect_type: type attendu (ex: dict), si fourni on vérifie isinstance.
    - label: nom lisible pour logs (ex: 'Ridge bundle OOS').
    """
    name = f" ({label})" if label else ""
    try:
        obj = load_any(primary_path)
        if expect_type and not isinstance(obj, expect_type):
            print(f"⚠️ Type inattendu{name} pour {primary_path}: {type(obj)} (attendu {expect_type}).")
        else:
            return obj
    except Exception as e:
        print(f"ℹ️ Impossible de charger {primary_path}{name} : {e}")

    for p in alt_paths:
        try:
            obj = load_any(p)
            print(f"ℹ️ Fallback utilisé{name} → {p}")
            if expect_type and not isinstance(obj, expect_type):
                print(f"⚠️ Type inattendu{name} pour {p}: {type(obj)} (attendu {expect_type}).")
            return obj
        except Exception as e:
            print(f"ℹ️ Fallback raté{name} → {p} : {e}")

    print(f"⚠️ Aucun fichier disponible{name} (essayé: {[primary_path, *alt_paths]})")
    return None

def try_read_meta(path):
    """Lit un CSV méta. Tente index_col=0 puis sans index si échec."""
    for use_index in (0, None):
        try:
            meta = pd.read_csv(path, index_col=use_index)
            # Si 1 colonne, exposer Series
            if isinstance(meta, pd.DataFrame) and meta.shape[1] == 1:
                meta = meta.iloc[:, 0]
            return meta
        except Exception as e:
            last_err = e
    print(f"⚠️ Impossible de lire {path} : {last_err}")
    return None

def safe_meta_print(meta, keys, title="Méta"):
    """Affiche les clés demandées sans lever d'erreur si elles manquent."""
    if meta is None:
        print("⚠️ Pas de méta disponible.")
        return
    if isinstance(meta, pd.Series):
        d = meta.to_dict()
    elif isinstance(meta, pd.DataFrame):
        d = meta.iloc[0].to_dict() if len(meta) else {}
    elif isinstance(meta, dict):
        d = meta
    else:
        print(f"(info) Type de méta inattendu: {type(meta)}")
        d = {}

    print(f"\n--- {title} (clé: valeur) ---")
    for k in keys:
        print(f"{k}: {d.get(k, None)}")

def _normalize_month_start(s):
    s = pd.to_datetime(s, errors="coerce")
    if isinstance(s, pd.Series):
        return s.dt.to_period("M").dt.to_timestamp(how="start")
    if isinstance(s, pd.DatetimeIndex):
        return s.to_period("M").to_timestamp(how="start")
    return pd.Timestamp(s).to_period("M").to_timestamp(how="start")

def ensure_ms_index_df(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.index = pd.to_datetime(out.index).to_period("M").to_timestamp(how="start")
    return out.sort_index().asfreq("MS")

def ensure_features_present(df: pd.DataFrame, features: list[str], target_col: str | None = None):
    missing = [c for c in (features or []) if c not in df.columns]
    if target_col is not None and target_col not in df.columns:
        missing.append(target_col)
    if missing:
        raise ValueError(f"Colonnes manquantes dans le DataFrame: {missing}")

def make_exp_from_loaded(*, models, features, train_periods, preprocs=None, step_months: int = 12) -> dict:
    if not isinstance(models, (list, tuple)) or len(models) == 0:
        raise ValueError("'models' doit être une liste non vue.")
    if not isinstance(features, (list, tuple)) or len(features) == 0:
        raise ValueError("'features' doit être une liste non vide.")
    if not isinstance(train_periods, (list, tuple)) or len(train_periods) != len(models):
        raise ValueError("'train_periods' doit avoir la même longueur que 'models'.")
    if preprocs is not None and len(preprocs) != len(models):
        raise ValueError("'preprocs' doit avoir la même longueur que 'models' si fourni.")
    return {
        "models":        list(models),
        "features":      list(features),
        "preprocs":      list(preprocs) if preprocs is not None else None,
        "train_periods": list(pd.to_datetime(pd.Index(train_periods))),
        "step_months":   int(step_months),
    }

def exp_from_linreg_bundle_no_refit(linreg_bundle: dict) -> dict:
    if not isinstance(linreg_bundle, dict):
        raise ValueError("linreg_bundle invalide (pas un dict).")
    params = linreg_bundle.get("params", {}) or {}
    features = params.get("features", linreg_bundle.get("features", None))
    if not features:
        raise ValueError("Features absentes dans le bundle (params.features).")
    models = linreg_bundle.get("models", None)
    if not isinstance(models, (list, tuple)) or len(models) == 0:
        raise RuntimeError(
            "Ce bundle ne contient pas la liste des modèles par fenêtre ('models'). "
            "Sauvegarde les modèles lors de l'apprentissage, puis recharge."
        )
    preprocs = linreg_bundle.get("preprocs", None)
    if preprocs is not None and len(preprocs) != len(models):
        raise RuntimeError("'preprocs' n'a pas la même longueur que 'models'.")
    train_periods = linreg_bundle.get("train_fit_dates", None)
    if train_periods is None or len(train_periods) != len(models):
        raise RuntimeError("'train_fit_dates' manquant ou de longueur différente de 'models'.")
    return {
        "models":        list(models),
        "features":      list(features),
        "preprocs":      list(preprocs) if preprocs is not None else None,
        "train_periods": list(pd.to_datetime(pd.Index(train_periods))),
        "step_months":   12,
    }

def extract_last_model_from_bundle(bundle: dict, model_key="models"):
    """Récupère le dernier modèle depuis un bundle, sinon None."""
    try:
        if isinstance(bundle, dict) and model_key in bundle and bundle[model_key]:
            return bundle[model_key][-1]
    except Exception as e:
        print(f"⚠️ Impossible d'extraire le dernier modèle: {e}")
    return None

In [55]:
from pathlib import Path

ARTIFACT_DIR = Path.cwd()   # ajuste si besoin

# ----------------------------
# AR(1) avec CI
# ----------------------------
AR1_BUNDLE     = ARTIFACT_DIR / "AR1_CI_h12_oos_bundle.pkl"
AR1_LAST_PKL   = ARTIFACT_DIR / "AR1_CI_last_trained_model.pkl"
AR1_LAST_META  = ARTIFACT_DIR / "AR1_CI_last_trained_model_meta.csv"

# (fallbacks si jamais tu as encore des anciens fichiers)
AR1_BUNDLE_OLD   = ARTIFACT_DIR / "AR1_h12_oos_bundle.pkl"
AR1_LAST_PKL_OLD = ARTIFACT_DIR / "AR1_last_trained_model.pkl"
AR1_LAST_META_OLD= ARTIFACT_DIR / "AR1_last_trained_model_meta.csv"

In [56]:
# -*- coding: utf-8 -*-
import os
import pandas as pd

# ==============================================================
# ✅ Helper CSV fallback (utilise try_read_meta existante)
# ==============================================================

def try_read_meta_with_fallbacks(primary_path, *alt_paths, label=None):
    name = f" ({label})" if label else ""
    for p in (primary_path, *alt_paths):
        if p is None:
            continue
        if not os.path.exists(str(p)):
            continue
        meta = try_read_meta(str(p))  # fonction existante
        if meta is not None:
            print(f"ℹ️ Meta CSV chargé{name} → {p}")
            return meta
    print(f"⚠️ Aucun CSV méta disponible{name}")
    return None

# ==============================================================
# ✅ Helpers bundle
# ==============================================================

def is_bundle(x):
    return isinstance(x, dict) and any(k in x for k in (
        "params", "models", "train_fit_dates",
        "oos", "oos_predictions", "forecasts"
    ))

def load_bundle_or_model(*paths, label=None):
    """
    Charge le premier fichier existant parmi paths via load_with_fallbacks.
    Retourne: (bundle, model)
    """
    obj = load_with_fallbacks(
        *[str(p) for p in paths if p is not None],
        expect_type=None,
        label=label
    )
    if obj is None:
        return None, None
    if is_bundle(obj):
        return obj, None
    return None, obj

# ==============================================================
# ✅ Chargement — AR(1) UNIQUEMENT (robuste NameError)
# ==============================================================

print("\n=== Chargement AR(1) ===")

AR1_BUNDLE_    = globals().get("AR1_BUNDLE")
AR1_LAST_PKL_  = globals().get("AR1_LAST_PKL")
AR1_PKL_       = globals().get("AR1_PKL")        # optionnel
AR1_LAST_META_ = globals().get("AR1_LAST_META")

ar1_bundle, ar1_model = load_bundle_or_model(
    AR1_BUNDLE_,
    AR1_LAST_PKL_,
    AR1_PKL_,
    label="AR1"
)

ar1_meta = try_read_meta_with_fallbacks(
    AR1_LAST_META_,
    label="AR1 meta"
)

# ==============================================================
# ✅ Récapitulatif minimal
# ==============================================================

print("\n=== Récap chargement AR(1) ===")
ok = (ar1_bundle is not None) or (ar1_model is not None)
print(f"AR1 : {'OK' if ok else '—'}")

if ar1_bundle is not None:
    print(" → bundle AR1 chargé")
elif ar1_model is not None:
    print(" → modèle AR1 chargé (hors bundle)")



=== Chargement AR(1) ===
ℹ️ Meta CSV chargé (AR1 meta) → d:\Portofolio Data science\Time Series\Explainable_AI_Forecast_and_explain_the_Unemployment_of_USA\3_notebook\AR1_CI_last_trained_model_meta.csv

=== Récap chargement AR(1) ===
AR1 : OK
 → bundle AR1 chargé


In [59]:
from pathlib import Path
import joblib


# --- chemins CI (priorité) + fallback anciens noms ---
AR1_LAST_PKL_CI   = Path(AR1_LAST_PKL)   # ex: "AR1_CI_last_trained_model.pkl"
AR1_LAST_META_CI  = Path(AR1_LAST_META)  # ex: "AR1_CI_last_trained_model_meta.csv"
AR1_BUNDLE_CI     = Path(AR1_BUNDLE)     # ex: "AR1_CI_h12_oos_bundle.pkl"

AR1_LAST_PKL_OLD  = Path("AR1_last_trained_model.pkl")
AR1_LAST_META_OLD = Path("AR1_last_trained_model_meta.csv")
AR1_BUNDLE_OLD    = Path("AR1_h12_oos_bundle.pkl")

print("Fichiers présents ?")
for name, p in {
    "AR1_LAST_PKL (CI)":  AR1_LAST_PKL_CI,
    "AR1_LAST_META (CI)": AR1_LAST_META_CI,
    "AR1_BUNDLE (CI)":    AR1_BUNDLE_CI,
    "AR1_LAST_PKL (OLD)": AR1_LAST_PKL_OLD,
    "AR1_LAST_META (OLD)":AR1_LAST_META_OLD,
    "AR1_BUNDLE (OLD)":   AR1_BUNDLE_OLD,
}.items():
    print(f"{name:<18} : {'OK' if p.exists() else 'ABSENT'}")

# -------- Chargement modèle (CI en priorité) --------
model_path = AR1_LAST_PKL_CI if AR1_LAST_PKL_CI.exists() else AR1_LAST_PKL_OLD
ar1 = load_any(str(model_path)) if model_path.exists() else None

def label(x):
    return x.__class__.__name__ if x is not None else None

print("\nLabel modèle :")
print("AR1 :", label(ar1))
print("Chargé depuis :", model_path if model_path.exists() else None)

Fichiers présents ?
AR1_LAST_PKL (CI)  : OK
AR1_LAST_META (CI) : OK
AR1_BUNDLE (CI)    : OK
AR1_LAST_PKL (OLD) : OK
AR1_LAST_META (OLD) : OK
AR1_BUNDLE (OLD)   : OK

Label modèle :
AR1 : AutoRegResultsWrapper
Chargé depuis : d:\Portofolio Data science\Time Series\Explainable_AI_Forecast_and_explain_the_Unemployment_of_USA\3_notebook\AR1_CI_last_trained_model.pkl


In [68]:
# Chargement du bundle AR(1) (CI en priorité)
ar1_bundle = load_with_fallbacks(
    str(AR1_BUNDLE), str(AR1_BUNDLE_OLD),
    expect_type=dict,
    label="AR1 bundle"
)

print("Keys:", ar1_bundle.keys() if ar1_bundle else None)

if ar1_bundle is not None:
    oos = ar1_bundle.get("oos_predictions")
    print("oos_predictions shape:", None if oos is None else oos.shape)
    print(oos.head() if oos is not None else "⚠️ oos_predictions manquant")

    # ✅ check des colonnes CI
    if oos is not None:
        cols = oos.columns.tolist()
        print("\nColonnes:", cols)

        has_ci = ("y_hat_lo_95" in cols and "y_hat_hi_95" in cols) or \
                 ("y_pred_lo_95" in cols and "y_pred_hi_95" in cols)

        print(f"\nCI présent ? {'✅ OUI' if has_ci else '❌ NON'}")


Keys: dict_keys(['oos_predictions', 'params', 'meta'])
oos_predictions shape: (741, 5)
        date    y_pred  y_true  y_pred_lo_95  y_pred_hi_95
0 1963-12-01 -0.080890     0.0           NaN           NaN
1 1964-01-01  0.141077    -0.1           NaN           NaN
2 1964-02-01  0.408114    -0.5           NaN           NaN
3 1964-03-01  0.242637    -0.3           NaN           NaN
4 1964-04-01  0.238955    -0.4           NaN           NaN

Colonnes: ['date', 'y_pred', 'y_true', 'y_pred_lo_95', 'y_pred_hi_95']

CI présent ? ✅ OUI


In [70]:
oos

,date,y_pred,y_true,y_pred_lo_95,y_pred_hi_95
0,1963-12-01,-0.080890,0.0,NaN,NaN
1,1964-01-01,0.141077,-0.1,NaN,NaN
2,1964-02-01,0.408114,-0.5,NaN,NaN
3,1964-03-01,0.242637,-0.3,NaN,NaN
4,1964-04-01,0.238955,-0.4,NaN,NaN
...,...,...,...,...,...
736,2025-04-01,0.130432,0.3,-0.498105,0.806477
737,2025-05-01,0.102350,0.2,-0.526187,0.778395
738,2025-06-01,0.131343,0.0,-0.497194,0.807388
739,2025-07-01,0.189149,0.0,-0.337558,0.865194


# Etablir de dataframe des résultats.

In [74]:
df_b = build_df_from_bundle(ar1_bundle, default_method="AR1")
print(df_b.head())
print(df_b.tail())

        date  true      pred  lo  hi method
0 1963-12-01   0.0 -0.080890 NaN NaN    AR1
1 1964-01-01  -0.1  0.141077 NaN NaN    AR1
2 1964-02-01  -0.5  0.408114 NaN NaN    AR1
3 1964-03-01  -0.3  0.242637 NaN NaN    AR1
4 1964-04-01  -0.4  0.238955 NaN NaN    AR1
          date  true      pred        lo        hi method
736 2025-04-01   0.3  0.130432 -0.498105  0.806477    AR1
737 2025-05-01   0.2  0.102350 -0.526187  0.778395    AR1
738 2025-06-01   0.0  0.131343 -0.497194  0.807388    AR1
739 2025-07-01   0.0  0.189149 -0.337558  0.865194    AR1
740 2025-08-01   0.1  0.132535 -0.244764  0.808580    AR1


In [75]:
n_ci = df_b[["lo","hi"]].notna().all(axis=1).sum()
print("Nb lignes avec CI non-NaN :", n_ci)

print("Premières lignes avec CI :")
print(df_b[df_b["lo"].notna() & df_b["hi"].notna()].head(3))

Nb lignes avec CI non-NaN : 705
Premières lignes avec CI :
         date  true      pred        lo        hi method
36 1966-12-01  -0.2 -0.445640 -1.641611 -0.537355    AR1
37 1967-01-01  -0.1 -0.408771 -1.604741 -0.479892    AR1
38 1967-02-01   0.0 -0.621048 -1.817018 -0.367516    AR1


# Filtrer

In [78]:
def filter_df_by_start_date(df: pd.DataFrame, start_date: str = "1990-01-01") -> pd.DataFrame:
    """
    Filtre un DataFrame standardisé (date, true, pred, method)
    en ne gardant que les observations à partir de `start_date`.
    Retourne un DataFrame trié + index propre.
    """
    out = df.copy()
    out["date"] = pd.to_datetime(out["date"], errors="coerce")

    out = (
        out[out["date"] >= pd.Timestamp(start_date)]
        .sort_values(["date", "method"])
        .reset_index(drop=True)
    )

    print(f"\n✅ Filtrage appliqué — période: {out['date'].min().date()} → {out['date'].max().date()} | n={len(out)}")
    print("Méthodes présentes :", sorted(out["method"].unique().tolist()))
    print("\nAperçu post-filtrage :")
    print(out.head(10))

    return out

In [79]:
df_pred_long_1990 = filter_df_by_start_date(
    df_b,
    start_date="1990-01-01"
)


✅ Filtrage appliqué — période: 1990-01-01 → 2025-08-01 | n=428
Méthodes présentes : ['AR1']

Aperçu post-filtrage :
        date  true      pred        lo        hi method
0 1990-01-01   0.0 -0.176432 -1.256478  0.232328    AR1
1 1990-02-01   0.1 -0.310379 -1.390425  0.098381    AR1
2 1990-03-01   0.2 -0.445518 -1.525564 -0.036223    AR1
3 1990-04-01   0.2 -0.111094 -1.191140  0.328677    AR1
4 1990-05-01   0.2 -0.244053 -1.324099  0.195719    AR1
5 1990-06-01  -0.1 -0.045497 -1.125543  0.423739    AR1
6 1990-07-01   0.3 -0.111754 -1.191800  0.357482    AR1
7 1990-08-01   0.5 -0.244439 -1.324485  0.224798    AR1
8 1990-09-01   0.6 -0.046283 -1.126329  0.611600    AR1
9 1990-10-01   0.6 -0.046434 -1.039138  0.612118    AR1


# Analyser la performance prédictive des modèles

## Comparaison des données de test et de prévision

In [85]:
# =========================
# Préparation pour plot_series (Plotly)
# =========================
import pandas as pd
from utilsforecast.plotting import plot_series

# Si tu veux tracer uniquement post-1990, remplace df_pred_long par df_pred_long_1990
df_plot = df_pred_long.copy()

# Assure ds en datetime (important)
df_plot["date"] = pd.to_datetime(df_plot["date"], errors="coerce")

df_obs = (
    df_plot
    .rename(columns={"date": "ds", "true": "y"})
    .assign(unique_id="UNRATE")
    [["unique_id", "ds", "y"]]
)

# Option: ne garder que les lignes où lo/hi existent (sinon intervalle incomplet au début)
df_plot_ci = df_plot[df_plot["lo"].notna() & df_plot["hi"].notna()].copy()

df_fcst = (
    df_plot_ci
    .rename(columns={
        "date": "ds",
        "pred": "AR",
        "lo": "AR-lo-95",
        "hi": "AR-hi-95",
    })
    .assign(unique_id="UNRATE")
    [["unique_id", "ds", "AR", "AR-lo-95", "AR-hi-95"]]
)

fig = plot_series(
    df=df_obs,
    forecasts_df=df_fcst,
    level=[95],
    engine="plotly",
).update_layout(height=400)

# Renommage légende
for trace in fig.data:
    if trace.name == "y":
        trace.name = "Unemployment rate (%)"
    elif trace.name == "AR":
        trace.name = "AutoRegressive (AR)"
    elif "level_95" in trace.name.lower():
        trace.name = "95% Prediction Interval"

fig.show()


In [ ]:
from utilsforecast.plotting import plot_series
import pandas as pd

# =========================
# Préparer les données (zoom recommandé)
# =========================
df_plot = df_pred_long_1990.copy()
df_plot["date"] = pd.to_datetime(df_plot["date"])

# Zoom : 2010 → aujourd’hui (idéal pour voir l’IC)
df_plot = df_plot[df_plot["date"] >= "1990-01-01"].copy()

df_obs = (
    df_plot.rename(columns={"date": "ds", "true": "y"})
           .assign(unique_id="UNRATE")[["unique_id", "ds", "y"]]
)

# Garder uniquement les dates avec IC
df_fcst = (
    df_plot[df_plot["lo"].notna() & df_plot["hi"].notna()]
    .rename(columns={
        "date": "ds",
        "pred": "AR",
        "lo": "AR-lo-95",
        "hi": "AR-hi-95",
    })
    .assign(unique_id="UNRATE")
    [["unique_id", "ds", "AR", "AR-lo-95", "AR-hi-95"]]
)

# =========================
# Plot
# =========================
fig = plot_series(
    df=df_obs,
    forecasts_df=df_fcst,
    level=[95],
    engine="plotly",
).update_layout(
    height=420,
    title="UNRATE — AR(1) Forecast with 95% Conformal Prediction Interval",
)

# =========================
# Stylisation fine
# =========================
for trace in fig.data:
    # Observé
    if trace.name == "y":
        trace.name = "Observed (UNRATE)"
        trace.update(line=dict(width=2))

    # Prévision AR
    elif trace.name == "AR":
        trace.name = "AR(1) forecast"
        trace.update(
            line=dict(width=3, dash="dash")
        )

    # Intervalle
    elif "level_95" in trace.name.lower():
        trace.name = "95% Prediction Interval"
        trace.update(
            opacity=0.20,
            line=dict(width=0)
        )

# Optionnel : grille plus discrète
fig.update_xaxes(showgrid=True, gridcolor="rgba(200,200,200,0.2)")
fig.update_yaxes(showgrid=True, gridcolor="rgba(200,200,200,0.2)")

fig.show()